# Week 1: Transactional Data Cleaning and Wrangling

## Objective

The objective of Week 1 is to prepare the raw transactional dataset for future cohort analysis.

Tasks:

- Load the transactional dataset
- Handle missing Customer IDs
- Remove refunded transactions
- Calculate Cohort Month for every customer

The cleaned dataset will be used in Week 2 for retention analysis.

In [1]:
import pandas as pd

In [2]:
import pandas as pd

df = pd.read_excel(
    "Online Retail.xlsx",
    engine="openpyxl"
)

df.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


# Dataset Overview

Before cleaning the dataset, we inspect its structure and identify potential data quality issues.

In [3]:
print("Dataset Shape:", df.shape)

df.info()

Dataset Shape: (541909, 8)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 541909 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   InvoiceNo    541909 non-null  object        
 1   StockCode    541909 non-null  object        
 2   Description  540455 non-null  object        
 3   Quantity     541909 non-null  int64         
 4   InvoiceDate  541909 non-null  datetime64[ns]
 5   UnitPrice    541909 non-null  float64       
 6   CustomerID   406829 non-null  float64       
 7   Country      541909 non-null  object        
dtypes: datetime64[ns](1), float64(2), int64(1), object(4)
memory usage: 33.1+ MB


# Missing User ID Analysis

CustomerID is required for cohort analysis because customers must be tracked across multiple transactions.

We first identify missing Customer IDs.

In [4]:
print(df['CustomerID'].isnull().sum())

135080


In [5]:
missing_customer_ids = df['CustomerID'].isnull().sum()

print(
    "Missing Customer IDs:",
    missing_customer_ids
)

Missing Customer IDs: 135080


# Remove Missing Customer IDs

Transactions without Customer IDs cannot be assigned to a cohort and are therefore removed.

In [6]:
df_clean = df.dropna(
    subset=['CustomerID']
)

print(
    "Rows After Removing Missing Customer IDs:",
    len(df_clean)
)

Rows After Removing Missing Customer IDs: 406829


# Refunded Transaction Analysis

In the Online Retail dataset, refunded transactions have Invoice Numbers beginning with the letter 'C'.

These transactions do not represent successful purchases and must be removed.

In [7]:
cancelled_orders = df_clean[
    df_clean['InvoiceNo']
    .astype(str)
    .str.startswith('C')
]

print(
    "Cancelled Transactions:",
    len(cancelled_orders)
)

Cancelled Transactions: 8905


# Remove Refunded Transactions

Refunded transactions are excluded from cohort analysis because they do not contribute to customer retention.

In [8]:
df_clean = df_clean[
    ~df_clean['InvoiceNo']
    .astype(str)
    .str.startswith('C')
]

print(
    "Rows After Removing Refunds:",
    len(df_clean)
)

Rows After Removing Refunds: 397924


# Calculate Cohort Month

A cohort is defined as the month in which a customer made their first purchase.

Customers who made their first purchase in the same month belong to the same cohort.

In [9]:
df_clean['InvoiceDate'] = pd.to_datetime(
    df_clean['InvoiceDate']
)

In [10]:
cohort_month = (
    df_clean.groupby('CustomerID')
    ['InvoiceDate']
    .min()
    .dt.to_period('M')
)

In [11]:
df_clean['CohortMonth'] = (
    df_clean['CustomerID']
    .map(cohort_month)
)

In [12]:
df_clean[
    [
        'CustomerID',
        'InvoiceDate',
        'CohortMonth'
    ]
].head(20)

,CustomerID,InvoiceDate,CohortMonth
0,17850.0,2010-12-01 08:26:00,2010-12
1,17850.0,2010-12-01 08:26:00,2010-12
2,17850.0,2010-12-01 08:26:00,2010-12
3,17850.0,2010-12-01 08:26:00,2010-12
4,17850.0,2010-12-01 08:26:00,2010-12
5,17850.0,2010-12-01 08:26:00,2010-12
6,17850.0,2010-12-01 08:26:00,2010-12
7,17850.0,2010-12-01 08:28:00,2010-12
8,17850.0,2010-12-01 08:28:00,2010-12
9,13047.0,2010-12-01 08:34:00,2010-12


# Handle Missing values (other columns)
Check all remaining nulls in df_clean.  

Drop any remaining null rows.

In [15]:
print("Missing values:\n", df_clean.isnull().sum())

Missing values:
 InvoiceNo      0
StockCode      0
Description    0
Quantity       0
InvoiceDate    0
UnitPrice      0
CustomerID     0
Country        0
CohortMonth    0
dtype: int64


In [16]:
df_clean = df_clean.dropna()
print("\nAfter droppinng nulls:", df_clean.shape)


After droppinng nulls: (397924, 9)


# Remove duplicated rows
Duplicate rows skews analysis by counting the same transaction twice. 

We identify and remove all duplicate entries.

In [17]:
print("Duplicates:", df_clean.duplicated().sum())

Duplicates: 5192


In [18]:
df_clean = df_clean.drop_duplicates()
print("After removing duplicates:", df_clean.duplicated().sum())

After removing duplicates: 0


# Validate Data Types
We ensure each column has the correct data type.

InvoiceDate must be a datetime, Quantity must be number, and CustomerID must be string

In [40]:
print("Initial Data types:\n", 
      df_clean.dtypes)

Initial Data types:
 InvoiceNo              object
StockCode              object
Description            object
Quantity                int64
InvoiceDate    datetime64[ns]
UnitPrice             float64
CustomerID             object
Country                object
CohortMonth         period[M]
dtype: object


In [24]:
df_clean["Quantity"] = pd.to_numeric(
    df_clean["Quantity"], errors="coerce")

df_clean["CustomerID"] = df_clean["CustomerID"].astype("int64").astype("str")

print("After Validation:\n", df_clean.dtypes)

After Validation:
 InvoiceNo              object
StockCode              object
Description            object
Quantity                int64
InvoiceDate    datetime64[ns]
UnitPrice             float64
CustomerID             object
Country                object
CohortMonth         period[M]
dtype: object


# Remove negatives values
Negative or zero values in quantity and price are invalid in sales data.

The records were removed to ensure accurate revenue calculations

In [38]:
print("Negative Quantity:", (df_clean["Quantity"]<0).sum())
print("Negative Unit Price:", (df_clean["UnitPrice"]<0).sum())

Negative Quantity: 0
Negative Unit Price: 0


In [36]:
df_clean = df_clean[
    df_clean["Quantity"]>0]
df_clean = df_clean[
    df_clean["UnitPrice"]>0]
print("Final shape:", df_clean.shape)

Final shape: (392692, 9)


# Export Cleaned Dataset

The cleaned dataset is exported and will be used in Week 2 to build the Cohort Retention Matrix.

In [39]:
df_clean.to_csv(
    "cleaned_retail_data.csv",
    index=False
)
print(
    "Cleaned dataset saved."
)

Cleaned dataset saved.
